# [Problem Name]

> **Time-box:** 45–60 minutes. Identify the hardest read/write trade-off early and spend your time there.

## Core requirements

1. _Requirement one._
2. _Requirement two._
3. _Requirement three._

## Stretch goals

- _Stretch goal one._
- _Stretch goal two._
- _Stretch goal three._

## Things the interviewer will probe

- **Pagination:** offset vs cursor. What breaks at scale?
- **Consistency:** eventual vs strong. Where can you afford lag?
- **Indexing:** which queries need indexes? What does a full-table scan cost?
- **Denormalization:** when is duplicating data worth it?
- **Idempotency:** which endpoints must be safe to retry?

---
## Setup

In [2]:
import json
import sqlite3

import pandas as pd
from fastapi import FastAPI
from fastapi.testclient import TestClient
from IPython.display import display
from pydantic import BaseModel

## Schema

Edit the SQL and re-run this cell to get a fresh in-memory database.

In [60]:
SCHEMA = """
CREATE TABLE riders (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL
);

CREATE TABLE drivers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    location VARCHAR(12)
);

CREATE TABLE rides (
    id INTEGER PRIMARY KEY,
    start_location VARCHAR(12) NOT NULL,
    end_location VARCHAR(12) NOT NULL,
    rider_id INTEGER NOT NULL REFERENCES riders(id),
    driver_id INTEGER DEFAULT NULL,
    fare_estimate_pence INTEGER,
    FOREIGN KEY (driver_id) REFERENCES drivers(id)
);


"""

conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript(SCHEMA)

print("Tables:", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()])

Tables: ['drivers', 'riders', 'rides']


## API

> After editing any cell below, re-run from **App** down through **Client**.

In [61]:
# ── App + models ──────────────────────────────────────────────────────────────
app = FastAPI(title="[Problem Name]")

class CreateRider(BaseModel):
    name: str

class CreateDriver(BaseModel):
    name: str

class CreateRide(BaseModel):
    start_location: str
    end_location: str

In [62]:
@app.post("/riders", status_code=201)
def create_rider(payload: CreateRider):
    with conn:
        cur = conn.execute("INSERT INTO riders(name) VALUES (?)", (payload.name,))
    return {"status": "ok"}

@app.post("/drivers", status_code=201)
def create_driver(payload: CreateDriver):
    with conn:
        cur = conn.execute("INSERT INTO drivers(name) VALUES (?)", (payload.name,))
    return {"status": "ok"}

In [63]:
@app.post("/rides")
def fare_estimate(payload: CreateRide, rider_id: int):
    with conn:
        conn.execute("INSERT INTO rides(start_location, end_location, rider_id) VALUES (?, ?, ?)", (payload.start_location, payload.end_location, rider_id))
    return {"status": "ok"}

In [64]:
# ── Client ────────────────────────────────────────────────────────────────────
client = TestClient(app, raise_server_exceptions=True)
print(client.get("/healthz").json())

{'detail': 'Not Found'}


## Helpers

In [65]:
def call(method: str, path: str, **kwargs):
    r = getattr(client, method)(path, **kwargs)
    body = r.json() if r.content else None
    print(f"{method.upper():6s} {path}  →  {r.status_code}")
    if body is not None:
        print(json.dumps(body, indent=2))
    return r


def df(table: str) -> pd.DataFrame:
    return pd.read_sql(f"SELECT * FROM {table}", conn)


def query(sql: str, *params) -> pd.DataFrame:
    return pd.read_sql(sql, conn, params=list(params) if params else None)


def show_all():
    names = [r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()]
    for name in names:
        count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
        print(f"\n── {name} ({count} rows) ──")
        display(pd.read_sql(f"SELECT * FROM {name}", conn))

## Demo

In [66]:
call("post", "/riders", json={"name": "Arnold"})
df("riders")

POST   /riders  →  201
{
  "status": "ok"
}


,id,name
0,1,Arnold


In [67]:
call("post", "/drivers", json={"name": "Van"})
df("drivers")

POST   /drivers  →  201
{
  "status": "ok"
}


,id,name,location
0,1,Van,None


In [ ]:
call("post", "/rides", json={"start_location": "0", "end_location": "1"}, params={"rider_id": 1})
df("rides")

POST   /rides  →  200
{
  "status": "ok"
}


<Response [200 OK]>

## All tables

In [ ]:
show_all()